In [1]:

import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import sys
%load_ext autoreload
%autoreload 2
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent)) 
from commom_utils.systems import *
from commom_utils.ode_system import ODESystem, check_system_ok, SyntheticDataGenerator
from commom_utils.system_config import create_system, SYSTEM_CONFIGS
from gauss_newton.utils import plot_solution
import matplotlib.pyplot as plt
from gauss_newton.gauss_newton_math import MultipleShooting, run_optimization
from typing import Callable
from experiments.data_utils import LogReaderV2, create_interval_batches, theta_to_physical

In [2]:
#path = Path("/home/iachichkanov/autotech/GaussNewton/experiments/CeedLateralIntensiveData.csv")
path = Path("/home/iachichkanov/autotech/GaussNewton/experiments/CeedEveron.csv")
data_storage = LogReaderV2(path)

Загружен default: 27999 строк из /home/iachichkanov/autotech/GaussNewton/experiments/CeedEveron.csv


In [3]:
data_storage.add_batch( "vx", use_jax_interp=True)
data_storage.add_batch( "vy", use_jax_interp=False)
data_storage.add_batch( "ay", use_jax_interp=False)
data_storage.add_batch( "yaw_rate",  use_jax_interp=False)
data_storage.add_batch( "steer", use_jax_interp=True)
data_storage.process_all()

Добавлен в очередь vx из default: time [0.000, 279.980], values [0.000, 14.826]
Добавлен в очередь vy из default: time [0.000, 279.980], values [-0.136, 0.157]
Добавлен в очередь ay из default: time [0.000, 279.980], values [-3.367, 3.035]
Добавлен в очередь yaw_rate из default: time [0.000, 279.980], values [-0.419, 0.416]
Добавлен в очередь steer из default: time [0.000, 279.980], values [-4.520, 4.082]

Общий t0 = 0.0
  используем jax
Обработан vx: норм. время [0.000, 279.980]
Обработан vy: норм. время [0.000, 279.980]
Обработан ay: норм. время [0.000, 279.980]
Обработан yaw_rate: норм. время [0.000, 279.980]
  используем jax
Обработан steer: норм. время [0.000, 279.980]
Общий временной интервал: [0.000, 279.980]


In [4]:
f_vx = data_storage.get_f_interp("vx")
f_vy = data_storage.get_f_interp("vy")
f_ay = data_storage.get_f_interp("ay")
f_steer = data_storage.get_f_interp("steer")
f_yaw_rate = data_storage.get_f_interp("yaw_rate")
t = data_storage.get_time("vx")
t1 = t[0]
t2 = t[-1]
t = np.linspace(t1, t2 - 10, 8000)
plt.plot(t, f_vx(t))
# plt.plot(t, f_vy(t))
#plt.plot(t, np.rad2deg(f_steer(t)))
# plt.plot(t, f_yaw_rate(t))

<Figure size 640x480 with 1 Axes>

In [5]:
fig = plt.figure(figsize=(20, 15))

plt.plot(t, f_ay(t))
plt.plot(t, f_vx(t)*f_vx(t)*np.tan(f_steer(t)/13)/2.65)

<Figure size 2000x1500 with 1 Axes>

In [6]:
fig = plt.figure(figsize=(20, 15))
plt.plot(t, f_vx(t)*np.tan(f_steer(t)/14)/2.65)
plt.plot(t, f_yaw_rate(t))

<Figure size 2000x1500 with 1 Axes>

In [7]:


def get_input_signals(t):
    return [f_vx(t), f_steer(t)]      




class DynamicModelRearAxle(ODESystem):
    def __init__(self, m, wheelbase, GR = None,  g=9.81,):
        self.m = m
        self.wheelbase = wheelbase
        self.g = g
        np = 4
        self.GR = GR
        if(self.GR is None):
            np +=1
        # состояния:  wz, vy_rear
        super().__init__(2, np, 2)

    def get_lateral_forces(self, rwa, vx, vy_rear, wz, theta):
        Cf_norm, Cr_norm, a_rel = theta[0], theta[1], theta[2]
        Cf = Cf_norm * self.m * self.g
        Cr = Cr_norm * self.m * self.g
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        # Скорость центра масс через заднюю ось
        vy_cm = vy_rear + b * wz
        alpha_f = rwa - (vy_cm + a * wz) / vx
        alpha_r = -(vy_cm - b * wz) / vx  
        Fyf = Cf * alpha_f
        Fyr = Cr * alpha_r
        return Fyf, Fyr, a, b

    def get_derivative(self, state, params, u):
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        
        GR = self.GR
        if(self.GR is None):
            GR = params[4]
        
        rwa = steering / GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # динамика центра масс (необходима для сил и ускорения)
        vy_cm = vy_rear + b * wz
        dvy_cm = (Fyf + Fyr) / self.m - vx * wz
        dwz = (a * Fyf - b * Fyr) / Iz

        # производная vy_rear
        dvy_rear = dvy_cm - b * dwz
        return ca.vertcat(dwz, dvy_rear)
    
    def calc_acc(self, state, params, u, d=0.0):
        """
        Вычисляет поперечное ускорение в точке, смещённой на d от центра масс.
        d > 0 – вперёд, к передней оси; d < 0 – назад.
        Состояние state (SX): [tau, psi, wz, vy_rear, rwa, rwa_dot, ...]
        """
        wz, vy_rear = state[0], state[1]

        vx, steering = u[0], u[1]
        GR = self.GR
        if(self.GR is None):
            GR = params[4]
        rwa = steering/GR
        # параметры
        a_rel = params[2]
        Iz_norm = params[3]


        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        Iz = Iz_norm * self.m * self.wheelbase**2

        Fyf, Fyr, a_calc, b_calc = self.get_lateral_forces(rwa, vx, vy_rear, wz, params)

        # Ускорение центра масс
        a_lat_cm = (Fyf + Fyr) / self.m

        # Угловое ускорение
        dwz = (a * Fyf - b * Fyr) / Iz

        # Ускорение в заданной точке
        a_lat = a_lat_cm + d * dwz

        return a_lat
    
    def observation(self, state: SX, theta: SX, u: SX):
        a_rel = theta[2]
        a = a_rel * self.wheelbase
        b = self.wheelbase - a
        a_lat = self.calc_acc(state, theta, u, d = -b)
        wz, vy = state[0], state[1]
        return ca.vertcat(a_lat, wz, vy)




config_dyn = {
    "class": DynamicModelRearAxle,
    "args": [1900, 2.65, None],                                 # wheelbase
    "c0": np.array([0.0]),      
    "theta_true": np.array([5,  5,  0.5,  0.4, 12.816658]), #
    "delta_theta": 0*np.array([4, 4.0, 0.2, 0.24]),
    "input_signal": get_input_signals,  #vx steering
}

t_batches = [t]


measured_batches = [np.vstack((f_ay(t), f_yaw_rate(t),  f_vy(t))).T]
system, c0, theta_init, _ = create_system(config_dyn)



In [8]:
system.get_input_signals(0)
system.m

1900

In [9]:
measured_batches[0]

array([[-0.16414366, -0.04022771,  0.02532205],
       [-0.38535924, -0.0362372 ,  0.0171868 ],
       [-0.2247545 , -0.0283731 ,  0.01937562],
       ...,
       [-0.21812041, -0.00222907, -0.01989222],
       [ 0.06583901, -0.00208799, -0.01511635],
       [-0.27200491, -0.002155  , -0.02403767]], shape=(8000, 3))

In [10]:
plt.plot(measured_batches[0][:140, :])

<Figure size 640x480 with 1 Axes>

In [11]:
gamma = np.zeros(3)
for i in range(3):
    gamma[i] = 1/np.std(measured_batches[0][:125, i])
    print(gamma[i])

6.439079179411109
145.63957212850372
104.06254140924739


In [15]:
# optimization_config.py (исправленная версия с доверительными интервалами)
import numpy as np
import matplotlib.pyplot as plt

class OptimizationConfig:
    """Конфигурация оптимизации для multiple shooting."""
    def __init__(self, n_obs):
        self.gamma = np.ones(n_obs)      # веса измерений
        self.gamma = gamma
        #assert len(self.gamma) == n_obs
        self.lambda_ = 0.005                # регуляризация Левенберга-Марквардта
        self.lambda_reg = 0.0001               # дополнительная регуляризация (отключена)
        self.n_iter = 30                   # количество итераций
        self.c0_cost = 1.0                  # вес начальной точки в интервале
        self.mu = 3*1e-2                     # начальный параметр для метода с множителями
        self.mu_dec = 0.5
        self.n_shoot = 10

def setup_problem(system, config, state_measured_batches,
                  t_eval_batches):
    problem = MultipleShooting(system, N_shoot=config.n_shoot, gamma=config.gamma,
                               c0_cost=config.c0_cost, use_jax=True)
    for state_meas, t_meas in zip(state_measured_batches, t_eval_batches):
        problem.add_batch(state_meas, t_meas)
    return problem



In [16]:
if __name__ == "__main__":

    config = OptimizationConfig(system.n_obs)
    # При необходимости переопределите параметры:
    problem = setup_problem(system, config,
                            measured_batches,  t_batches)
    theta0 = theta_init   # theta определена ранее

    theta_full = problem.make_full_theta(theta0, n_iter = 10)
    theta_hist, r_meas_hist, r_cont_hist, theta_full, ci_low_hist, ci_high_hist = run_optimization(
        problem, config, theta_full, system
    )


  J nnz: 167879, J_G nnz: 108
Iter   0 | R_meas: 5.685e+00 | R_cont: 4.937e-04 | mu: 3.00e-02
  Iter time: 0.865s
Iter   1 | R_meas: 3.079e+00 | R_cont: 4.528e-04 | mu: 1.50e-02
  Iter time: 1.437s
Iter   2 | R_meas: 2.969e+00 | R_cont: 2.661e-04 | mu: 7.50e-03
  Iter time: 1.100s
Iter   3 | R_meas: 2.960e+00 | R_cont: 1.453e-04 | mu: 3.75e-03
  Iter time: 1.348s
Iter   4 | R_meas: 2.959e+00 | R_cont: 1.273e-04 | mu: 1.87e-03
  Iter time: 1.125s
Iter   5 | R_meas: 2.959e+00 | R_cont: 1.376e-04 | mu: 9.37e-04
  Iter time: 1.284s
Iter   6 | R_meas: 2.959e+00 | R_cont: 1.224e-04 | mu: 4.69e-04
  Iter time: 1.785s
Iter   7 | R_meas: 2.959e+00 | R_cont: 1.218e-04 | mu: 2.34e-04
  Шаг отклонён (cost 7.100e+04 > 7.100e+04), mu сохранён 2.34e-04
  Iter time: 1.315s
Iter   8 | R_meas: 2.959e+00 | R_cont: 1.218e-04 | mu: 4.69e-04
  Iter time: 1.200s
Iter   9 | R_meas: 2.959e+00 | R_cont: 1.159e-04 | mu: 2.34e-04
  Iter time: 1.019s
Iter  10 | R_meas: 2.959e+00 | R_cont: 1.059e-04 | mu: 1.17e-04


In [14]:
theta_full

array([ 6.98544929e+00,  7.07778970e+00,  3.96335840e-01,  1.22742690e-01,
       -3.67060009e-03, -1.93251665e-02, -2.24567164e-01,  6.56156433e-02,
        2.30374081e-03, -2.52832201e-02,  2.94276675e-02, -7.76061401e-03,
        1.25440558e-02, -2.11474278e-02, -2.34070588e-02,  4.19144336e-03,
        2.15960017e-03, -6.25991569e-03,  2.42231381e-02,  7.86789942e-03,
        2.24734731e-01, -1.13995221e-02, -2.59395043e-02, -7.27903314e-04])

In [22]:
theta_hist[-1]


array([ 5.83155892e+00,  6.54352363e+00,  3.76644227e-01,  1.34874019e-01,
        1.17022730e+01, -2.48276874e-02, -2.18620881e-02,  1.03480330e-02,
       -2.53143215e-01, -2.77936180e-02,  1.65158092e-01,  7.69008124e-02,
       -2.78997779e-01,  5.70736805e-02, -1.87878089e-01,  1.41452956e-01,
       -2.71193584e-01, -1.07029595e-01,  1.64751345e-01,  4.78062377e-02,
       -5.78385003e-02, -5.98618266e-02,  6.04729394e-02,  5.56390765e-02,
       -3.83882978e-02])

In [17]:

plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=True,
    plot_measurements = 1,
    r_meas_hist=r_meas_hist,
    r_cont_hist=r_cont_hist,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    #param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.np)]
)

In [15]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
theta_hist[-1][:4], config  # without GR estiname everon GR = 14


(array([7.8831957 , 4.35951889, 0.26104269, 0.09832541]),
 {'Cf': np.float64(146934.88465657164),
  'Cr': np.float64(81257.07257371504),
  'a': np.float64(0.6917631354326136),
  'b': np.float64(1.9582368645673864),
  'Iz': np.float64(1311.9313320250749)})

In [15]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
theta_hist[-1][:4], config  # without GR estiname poligon GR = 14


(array([9.38527972, 5.14373912, 0.2686535 , 0.13674789]),
 {'Cf': np.float64(174932.2286898698),
  'Cr': np.float64(95874.1534973679),
  'a': np.float64(0.7119317686422532),
  'b': np.float64(1.9380682313577466),
  'Iz': np.float64(1824.592896250395)})

In [ ]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
theta_hist[-1][:5], config  # with GR estiname everon 

(array([ 8.59617841,  4.19628865,  0.22535619,  0.12338291, 12.57116362]),
 {'Cf': np.float64(160224.16934787543),
  'Cr': np.float64(78214.62418735182),
  'a': np.float64(0.5971938903921589),
  'b': np.float64(2.052806109607841),
  'Iz': np.float64(1646.2672688432294)})

In [16]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
config,  theta_hist[-1][4] # # with GR estiname poligon 


({'Cf': np.float64(50297.36211263032),
  'Cr': np.float64(148492.53631984003),
  'a': np.float64(1.386563156392004),
  'b': np.float64(1.263436843607996),
  'Iz': np.float64(971.9515901503048)},
 np.float64(12.289102793083439))

In [35]:
theta_to_physical(np.array([6.98544929, 7.0777897 , 0.39633584, 0.12274269]), m = 1900, L =2.65)

{'Cf': np.float64(130201.78931631),
 'Cr': np.float64(131922.92221830002),
 'a': np.float64(1.050289976),
 'b': np.float64(1.599710024),
 'Iz': np.float64(1637.7250269975)}

In [23]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
config

{'Cf': np.float64(130201.78923980573),
 'Cr': np.float64(131922.92224989113),
 'a': np.float64(1.050289984608812),
 'b': np.float64(1.5997100153911878),
 'Iz': np.float64(1637.725005354941)}

In [ ]:
config = OptimizationConfig(system.n_obs)
problem = setup_problem(system, config,
                    measured_batches,  t_batches)
theta_full = problem.make_full_theta(np.array([6.98544929, 7.0777897 , 0.39633584, 0.12274269]), n_iter = 10)
theta_hist = [theta_full]


plot_solution(
    problem = problem, 
    theta_hist = theta_hist,
    plot_xy=1,
    plot_theta=True,
    plot_trajectory=0,
    plot_true_solution=False,
    plot_residuals=0,
    plot_measurements = 1,
    index=-1,
    theta_true=None,
    fontsize = 20,
    # ci_low_hist=ci_low_hist,    # <-- добавить
    # ci_high_hist=ci_high_hist,  # <-- добавить
    state_names=['a_y', 'w_z', 'vy'],
    #param_names=['C<sub>f</sub>', 'C<sub>r</sub>', 'a<sub>rel</sub>', 'I<sub>norm</sub>']
   # param_names=[f'θ_{i}' for i in range( system.np)]
)

array([ 6.98544929,  7.0777897 ,  0.39633584,  0.12274269, -0.02967119,
       -0.01369388,  0.01130482, -0.26439271, -0.02904051,  0.16832663,
        0.08101964, -0.27670152,  0.06255787, -0.18000973,  0.14085677,
       -0.25640628, -0.10166432,  0.15709166,  0.04873055, -0.04910102,
       -0.05439899,  0.05502061,  0.0533996 , -0.03222616])

In [ ]:
(1900/2.65)*(config["a"]/config['Cf'] - config["b"]/config['Cr'])

np.float64(-0.0037202467327500568)

In [ ]:
(1900/2.65)*(1.2333664515275726 /59406.34085264137- 1.4166335484724273/133076.911608564)

0.007253199525996578

In [ ]:
0.007253199525996578*25**2

4.533249703747861

In [ ]:
config = theta_to_physical(theta_hist[-1][:4], 1900, 2.65)
config


{'Cf': np.float64(59406.34085264137),
 'Cr': np.float64(133076.911608564),
 'a': np.float64(1.2333664515275726),
 'b': np.float64(1.4166335484724273),
 'Iz': np.float64(1092.2923296101428)}

In [ ]:
theta_hist[0][:5], theta_hist[-1][:5]

(array([ 4.49955229,  7.92448103,  0.54657406,  0.16234435, 12.816658  ]),
 array([ 2.76587496,  8.19576105,  0.53505593,  0.08770295, 12.37426532]))

In [ ]:
theta_hist[0][:5], theta_hist[-1][:5]

(array([ 3.0642791 ,  7.26640357,  0.6031844 ,  0.31198512, 14.        ]),
 array([ 4.49955229,  7.92448103,  0.54657406,  0.16234435, 12.81665875]))

In [ ]:
config = theta_to_physical(theta_hist[-1][:5], 1700, 2.65)
config


{'Cf': np.float64(69550.88993644534),
 'Cr': np.float64(225649.61137671844),
 'a': np.float64(1.7656617581599081),
 'b': np.float64(0.8843382418400918),
 'Iz': np.float64(2649.4601012946537),
 'steering_ratio': np.float64(12.684747488766583)}

In [ ]:
theta_to_physical()

array([29986.9867735 ,  3040.09843302])